# Stage 2 - v4: EfficientNet-B3 (Interpolated Focal Loss)

## 1. Giris ve Problem Tanimi

Bu notebook, inme arter siniflandirmasi projesinin **v4** iterasyonudur.

### Onceki Sonuclar
* **v2 (Aggressive Focal):** ACA Recall **0.8393** (en iyi), MCA Recall 0.8676. Agresif agirliklar (ACA=4.13) MCA'yi dusurdu.
* **v3 (Softened Focal):** ACA Recall 0.7857 (gerileme), MCA Recall 0.9185. Power=0.5 ile yumusatma ASIRI yapildi, ACA kayboldu.
* **Tahterevalli Etkisi:** v2 ile v3 arasinda ACA ve MCA ters yonde hareket etti.

### v4 Stratejisi: v2-v3 Interpolasyonu
v2'nin ACA basarisini korurken v3'un denge avantajindan faydalanmak icin:
1. **Gamma: 1.8** (v2=2.0, v3=1.5 arasi, v2'ye yakin) — Hard-example mining'i korur, MCA'yi biraz rahatlatir.
2. **Weight Power: 0.75** (v2=1.0, v3=0.5 arasi) — Agirliklar v2'ye daha yakin tutuldu cunku ACA henuz hedefe ulasmadi.

### Tahmini Class Weights
* ACA: ~1.78 (v2: 4.13, v3: 1.53)
* MCA: ~0.36 (v2: 0.47, v3: 0.52)
* PCA: ~0.86 (v2: 1.59, v3: 0.95)

### Hedef
ACA Recall >= 0.84 (v2 seviyesi veya ustu), MCA Recall >= 0.87 (v2'den iyi)

## Bolum 1: Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tqdm import tqdm
from collections import Counter

# Seed Setting
SEED = 42
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# Device Config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Bolum 2: Sabitler ve Hiperparametreler

In [ ]:
# --- PATHS ---
GCS_DATA_PATH = 'gs://stroke-detection/data/stroke_dataset/stroke_dataset/'

# --- CLASS INFO ---
CLASS_NAMES = ['ACA', 'MCA', 'PCA']
NUM_CLASSES = len(CLASS_NAMES)

# --- HYPERPARAMETERS (v4) ---
IMG_SIZE = 300
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100

# v4: v2 (2.0) ve v3 (1.5) arasi, v2'ye yakin
GAMMA = 1.8

# v4: v2 (1.0) ve v3 (0.5) arasi
WEIGHT_POWER = 0.75

# Scheduler
T_0 = 10
T_MULT = 2

# Split Ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f'=== Stage 2 - v4 Parametreleri ===')
print(f'Model: EfficientNet-B3')
print(f'Loss: FOCAL LOSS (Gamma={GAMMA})')
print(f'Weight Strategy: Power-Law Smoothing (Power={WEIGHT_POWER})')
print(f'Scheduler: CosineAnnealingWarmRestarts')

## Bolum 2.5: GCS Data Setup

In [ ]:
import os, subprocess
from pathlib import Path

if os.path.exists('/kaggle/input/stroke-images/flattened_images'):
    STROKE_IMAGES_DIR = '/kaggle/input/stroke-images/flattened_images'
    print('Ortam: Kaggle')
elif os.path.exists('/tmp/data/stroke_dataset/stroke_dataset') and len(os.listdir('/tmp/data/stroke_dataset/stroke_dataset')) >= 3:
    STROKE_IMAGES_DIR = '/tmp/data/stroke_dataset/stroke_dataset'
    print('Ortam: Vertex AI — Veri zaten mevcut')
else:
    print(f'GCS\'den veri indiriliyor: {GCS_DATA_PATH}')
    os.makedirs('/tmp/data', exist_ok=True)
    result = subprocess.run(['gsutil', '-m', 'cp', '-r', GCS_DATA_PATH, '/tmp/data/'], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'GCS download failed: {result.stderr}')
    STROKE_IMAGES_DIR = '/tmp/data/stroke_dataset/stroke_dataset'

for cls in ['ACA', 'MCA', 'PCA']:
    count = len(list(Path(STROKE_IMAGES_DIR, cls).glob('*'))) if Path(STROKE_IMAGES_DIR, cls).exists() else 0
    print(f'  {cls}: {count} goruntu')

## Bolum 3: Veri Hazirligi

In [ ]:
def collect_stroke_image_paths(stroke_dir, class_names):
    image_paths = []
    labels = []
    for idx, class_name in enumerate(class_names):
        class_dir = Path(stroke_dir) / class_name
        if not class_dir.exists(): continue
        
        extensions = ['*.png', '*.jpg', '*.jpeg']
        class_images = []
        for ext in extensions:
            class_images.extend(list(class_dir.glob(ext)))
            
        for img_path in class_images:
            image_paths.append(str(img_path))
            labels.append(idx)
    return np.array(image_paths), np.array(labels)

all_image_paths, all_labels = collect_stroke_image_paths(STROKE_IMAGES_DIR, CLASS_NAMES)

# Stratified Split
X_temp, X_test, y_temp, y_test = train_test_split(
    all_image_paths, all_labels, test_size=TEST_RATIO, stratify=all_labels, random_state=SEED
)
val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_adjusted, stratify=y_temp, random_state=SEED
)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')
print(f'Train class distribution: {Counter(y_train)}')

## Bolum 4: Augmentation & Dataset

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.ElasticTransform(alpha=50, sigma=50 * 0.05, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class StrokeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = int(self.labels[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        return image, label

train_dataset = StrokeDataset(X_train, y_train, transform=train_transform)
val_dataset = StrokeDataset(X_val, y_val, transform=val_transform)
test_dataset = StrokeDataset(X_test, y_test, transform=val_transform)

## Bolum 5: WeightedRandomSampler & Interpolated Class Weights (v4)

In [ ]:
# 1. Sampler (Ayni — Batch icinde sinif dengeleme)
class_counts_train = Counter(y_train)
class_weights_sampler = {c: 1.0 / count for c, count in class_counts_train.items()}
sample_weights = [class_weights_sampler[int(label)] for label in y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# 2. Loss Icin Interpolated Agirliklar (v4)
# Formula: Weight = (Total / (Num_Classes * Count)) ^ WEIGHT_POWER
total_samples = sum(class_counts_train.values())
class_weights_loss = []

print('\nLoss Fonksiyonu Icin Sinif Agirliklari (Interpolated v4):')
for i in range(NUM_CLASSES):
    count = class_counts_train[i]
    raw_weight = total_samples / (NUM_CLASSES * count)
    soft_weight = raw_weight ** WEIGHT_POWER
    class_weights_loss.append(soft_weight)
    print(f'  {CLASS_NAMES[i]}: Raw={raw_weight:.2f} -> Power({WEIGHT_POWER})={soft_weight:.4f}')

# Normalizasyon: Toplam = NUM_CLASSES
weight_sum = sum(class_weights_loss)
class_weights_loss = [w * NUM_CLASSES / weight_sum for w in class_weights_loss]

print('\nNormalize Edilmis Final Agirliklar:')
for i, w in enumerate(class_weights_loss):
    print(f'  {CLASS_NAMES[i]}: {w:.4f}')

print(f'\nKarsilastirma:')
print(f'  v2 ACA weight: 4.1346 | v3 ACA weight: 1.5334 | v4 ACA weight: {class_weights_loss[0]:.4f}')
print(f'  v2 MCA weight: 0.4700 | v3 MCA weight: 0.5170 | v4 MCA weight: {class_weights_loss[1]:.4f}')

class_weights_tensor = torch.FloatTensor(class_weights_loss).to(device)

## Bolum 6: Focal Loss (Gamma=1.8)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print(f'Focal Loss tanimlandi (Gamma={GAMMA}).')

## Bolum 7: Model & Optimizer

In [ ]:
def create_model(num_classes):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(num_features, num_classes)
    )
    return model

model = create_model(NUM_CLASSES).to(device)

# Loss: Interpolated Weights + Gamma=1.8
criterion = FocalLoss(alpha=class_weights_tensor, gamma=GAMMA)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=T_0,
    T_mult=T_MULT,
    eta_min=1e-6
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')

## Bolum 8: Egitim Fonksiyonlari

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    return running_loss / total, correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = running_loss / total
    accuracy = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, accuracy, macro_f1, np.array(all_preds), np.array(all_labels)

## Bolum 9: Egitim Dongusu

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0.0
CHECKPOINT_PATH = 'best_model_v4.pth'

print('Stage 2 - v4 Egitimi Basliyor (Interpolated Focal Loss)...')
print(f'Gamma={GAMMA}, Weight Power={WEIGHT_POWER}')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, device)
    
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | LR: {current_lr:.2e}')
    print(f'  Train Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%')
    print(f'  Val   Loss: {val_loss:.4f} | Acc: {val_acc*100:.2f}% | F1: {val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        print(f'  >>> Iyilesme! (F1: {best_val_f1:.4f} -> {val_f1:.4f}). Model kaydediliyor...')
        best_val_f1 = val_f1
        torch.save(model.state_dict(), CHECKPOINT_PATH)
    
    print('-' * 60)

model.load_state_dict(torch.load(CHECKPOINT_PATH))
print(f'\nEgitim tamamlandi. En iyi model (Val F1: {best_val_f1:.4f}) yuklendi.')

## Bolum 10: Egitim Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].set_xlabel('Epoch')

axes[2].plot(history['val_f1'], label='Val Macro F1')
axes[2].set_title('Validation Macro F1')
axes[2].legend()
axes[2].set_xlabel('Epoch')

plt.suptitle('v4 Training History (Gamma=1.8, Power=0.75)', fontsize=14)
plt.tight_layout()
plt.show()

## Bolum 11: Test Degerlendirmesi

In [ ]:
print('=== Stage 2 - v4 Test Degerlendirmesi ===')

test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, device)

print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Macro F1: {test_f1:.4f}')

print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - v4 Model (Interpolated Focal)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

# Per-class metrics
recall_per_class = recall_score(test_labels, test_preds, average=None)
precision_per_class = precision_score(test_labels, test_preds, average=None)
f1_per_class = f1_score(test_labels, test_preds, average=None)

print('\n=== v4 Sonuclari ===')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls}: Precision={precision_per_class[i]:.4f}, Recall={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}')

macro_recall = recall_score(test_labels, test_preds, average='macro')
macro_precision = precision_score(test_labels, test_preds, average='macro')
print(f'\n  Macro Precision: {macro_precision:.4f}')
print(f'  Macro Recall: {macro_recall:.4f}')
print(f'  Macro F1: {test_f1:.4f}')
print(f'  Test Accuracy: {test_acc*100:.2f}%')

## Bolum 12: Karsilastirma ve Sonuc

In [ ]:
print('=' * 60)
print('VERSIYONLAR ARASI KARSILASTIRMA')
print('=' * 60)

aca_recall = recall_per_class[0]
mca_recall = recall_per_class[1]
pca_recall = recall_per_class[2]

print(f'{"Version":<12} {"Test Acc":<10} {"Macro F1":<10} {"ACA Rec":<10} {"MCA Rec":<10} {"PCA Rec":<10}')
print('-' * 62)
print(f'{"baseline":<12} {"89.02%":<10} {"0.8400":<10} {"0.7679":<10} {"0.8982":<10} {"0.9103":<10}')
print(f'{"v1":<12} {"90.17%":<10} {"0.8449":<10} {"0.7321":<10} {"0.9226":<10} {"0.8966":<10}')
print(f'{"v2":<12} {"88.29%":<10} {"0.8401":<10} {"0.8393":<10} {"0.8676":<10} {"0.9517":<10}')
print(f'{"v3":<12} {"90.03%":<10} {"0.8479":<10} {"0.7857":<10} {"0.9185":<10} {"0.8828":<10}')
print(f'{"v4":<12} {test_acc*100:.2f}%     {test_f1:.4f}     {aca_recall:.4f}     {mca_recall:.4f}     {pca_recall:.4f}')
print('-' * 62)

# Hedef kontrolu
print('\n=== HEDEF KONTROLU ===')
targets = {
    'ACA Recall >= 0.85': aca_recall >= 0.85,
    'MCA Recall >= 0.85': mca_recall >= 0.85,
    'PCA Recall >= 0.85': pca_recall >= 0.85,
    'Macro Recall >= 0.83': macro_recall >= 0.83,
    'Macro F1 >= 0.80': test_f1 >= 0.80,
}
for target, met in targets.items():
    status = 'ULASILDI' if met else 'ULASILAMADI'
    print(f'  {target}: {status}')

print('\n=== SONUC ===')
print(f'v4 Parametreleri: Gamma={GAMMA}, Weight Power={WEIGHT_POWER}')
print(f'v4 ACA Recall: {aca_recall:.4f} (v2: 0.8393, v3: 0.7857)')
print(f'v4 MCA Recall: {mca_recall:.4f} (v2: 0.8676, v3: 0.9185)')
print(f'v4 Macro F1:   {test_f1:.4f} (v2: 0.8401, v3: 0.8479)')